In [1]:
import pandas as pd
from src.data import DATA_DIR_INTERIM

from src.io import load_qrels_from_path, read_metadata
from topic_gen.evaluate import MetaExperiment
from topic_gen.evaluate.io import load_from_irds
from topic_gen.evaluate.measures_agreement import CohenKappa, AreaUnderReceiver, MeanAverageError
from topic_gen.evaluate.utils import QrelsTransformer
import ir_datasets

from topic_gen import logger
logger.setLevel("DEBUG")

# Binary

In [2]:
BASE_DIR = DATA_DIR_INTERIM / "dl19" / "qrels-dl19-reference"
experiments = load_qrels_from_path(
    BASE_DIR, binarize_qrels=1, drop_relevance_values=999)

Error loading experiment from 2025-12-10_06:31:40: [Errno 2] No such file or directory: '/workspaces/conf26-generating-topics/data/interim/dl19/qrels-dl19-reference/2025-12-10_06:31:40/qrels.csv.gz'


In [3]:
baseline = load_from_irds(
    "msmarco-passage/trec-dl-2019/judged", binarize_qrels=1)

In [4]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), AreaUnderReceiver(), MeanAverageError()],
    filter_qrels=True
)

In [5]:
res = meta_exp.evaluate()

In [6]:
metadata = read_metadata(BASE_DIR)

[topic_gen] [WARNING] (io.py:45) Metadata not found for result 2025-12-10_06:31:40, skipping...


In [7]:
df = pd.DataFrame(res)
missing = (
    df[["name", "missing_qrels"]].drop_duplicates(
    ).set_index("name").to_dict()["missing_qrels"]
)

df = df.pivot(index="name", columns="measure", values="value").reset_index()

df = df.merge(metadata, left_on="name", right_on="date")
df["missing"] = df["name"].map(missing)

In [9]:
df[["model", "prompt", "CohenKappa", "MeanAverageError",
    "AreaUnderReceiver", "missing"]].round(2)

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver,missing
0,qwen3-14B-no-think,qrel_zeroshot_bing,0.35,0.34,0.74,18
1,gpt-oss-20B,qrel_zeroshot_bing,0.37,0.33,0.73,26
2,qwen3-30B-no-think,qrel_zeroshot_bing,0.23,0.42,0.72,0
3,gpt-oss-120B-ollama,qrel_zeroshot_bing,0.38,0.33,0.73,1
4,deepseek-V3.2,qrel_zeroshot_bing,0.51,0.25,0.77,0


# Binary Replace Missing

In [9]:
BASE_DIR = DATA_DIR_INTERIM / "dl19" / "qrels-dl19-reference"
experiments = load_qrels_from_path(
    BASE_DIR, binarize_qrels=1, replace_label_mapping={999: 0})

Error loading experiment from 2025-12-10_06:31:40: [Errno 2] No such file or directory: '/workspaces/conf26-generating-topics/data/interim/dl19/qrels-dl19-reference/2025-12-10_06:31:40/qrels.csv.gz'


In [10]:
baseline = load_from_irds(
    "msmarco-passage/trec-dl-2019/judged", binarize_qrels=1)

In [11]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), AreaUnderReceiver(), MeanAverageError()],
    filter_qrels=True
)

In [12]:
res = meta_exp.evaluate()

In [13]:
metadata = read_metadata(BASE_DIR)

[topic_gen] [WARNING] (io.py:45) Metadata not found for result 2025-12-10_06:31:40, skipping...


In [14]:
df = pd.DataFrame(res)
missing = (
    df[["name", "missing_qrels"]].drop_duplicates(
    ).set_index("name").to_dict()["missing_qrels"]
)

df = df.pivot(index="name", columns="measure", values="value").reset_index()

df = df.merge(metadata, left_on="name", right_on="date")
df["missing"] = df["name"].map(missing)

In [15]:
df[["name", "model", "prompt", "CohenKappa", "MeanAverageError",
    "AreaUnderReceiver", "missing"]].round(2)

,name,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver,missing
0,2025-12-05_15:15:10,qwen3-14B-no-think,qrel_zeroshot_bing,0.35,0.34,0.74,0
1,2025-12-07_16:20:49,gpt-oss-20B,qrel_zeroshot_bing,0.36,0.33,0.73,0
2,2025-12-09_06:32:42,qwen3-30B-no-think,qrel_zeroshot_bing,0.23,0.42,0.72,0
3,2025-12-09_07:24:09,gpt-oss-120B-ollama,qrel_zeroshot_bing,0.38,0.33,0.73,0
4,2025-12-10_06:31:57,deepseek-V3.2,qrel_zeroshot_bing,0.51,0.25,0.77,0


## Graded

In [10]:
BASE_DIR = DATA_DIR_INTERIM / "dl19" / "qrels-dl19-reference"
experiments = load_qrels_from_path(BASE_DIR, replace_label_mapping={999: 0})

Error loading experiment from 2025-12-10_06:31:40: [Errno 2] No such file or directory: '/workspaces/conf26-generating-topics/data/interim/dl19/qrels-dl19-reference/2025-12-10_06:31:40/qrels.csv.gz'


In [11]:
baseline = load_from_irds("msmarco-passage/trec-dl-2019/judged")

In [14]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), MeanAverageError()],
    filter_qrels=True
)

In [15]:
res = meta_exp.evaluate()

In [16]:
df = pd.DataFrame(res)
missing = (
    df[["name", "missing_qrels"]].drop_duplicates(
    ).set_index("name").to_dict()["missing_qrels"]
)

df = df.pivot(index="name", columns="measure", values="value").reset_index()

df = df.merge(metadata, left_on="name", right_on="date")
df["missing"] = df["name"].map(missing)

In [18]:
df[["name", "model", "prompt", "CohenKappa", "MeanAverageError","missing"]].round(2)

,name,model,prompt,CohenKappa,MeanAverageError,missing
0,2025-12-05_15:15:10,qwen3-14B-no-think,qrel_zeroshot_bing,0.25,0.68,0
1,2025-12-07_16:20:49,gpt-oss-20B,qrel_zeroshot_bing,0.24,0.71,0
2,2025-12-09_06:32:42,qwen3-30B-no-think,qrel_zeroshot_bing,0.09,1.08,0
3,2025-12-09_07:24:09,gpt-oss-120B-ollama,qrel_zeroshot_bing,0.25,0.68,0
4,2025-12-10_06:31:57,deepseek-V3.2,qrel_zeroshot_bing,0.29,0.69,0
